In [2]:
from pathlib import Path
import pandas as pd
import scipy.sparse
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from surprise import SVD, Dataset, Reader

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT = Path().resolve().parent
DATA = ROOT / "data"
SRC  = ROOT / "src"
SRC.mkdir(exist_ok=True)

# ── 1. Load Data ───────────────────────────────────────────────────────────────
movies  = pd.read_csv(DATA / "movies.csv")
ratings = pd.read_csv(DATA / "ratings.csv")

print(f"Movies  : {len(movies):,}")
print(f"Ratings : {len(ratings):,}")
print(f"Users   : {ratings['userId'].nunique():,}")

# ── 2. Clean Genres (for TF-IDF fallback) ─────────────────────────────────────
movies['genres_clean'] = (
    movies['genres']
    .str.replace('|', ' ', regex=False)
    .str.replace('-', '', regex=False)
)
movies = movies[movies['genres'] != '(no genres listed)'].reset_index(drop=True)

# ── 3. Build & Save TF-IDF Matrix ─────────────────────────────────────────────
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genres_clean'])

scipy.sparse.save_npz(SRC / "tfidf_matrix.npz", tfidf_matrix)
movies.to_pickle(SRC / "movies.pkl")
print(f"\n✅ TF-IDF artifacts saved")

# ── 4. Train SVD Model ────────────────────────────────────────────────────────
# SVD (Singular Value Decomposition) is matrix factorization.
# It learns latent factors — hidden patterns like "this movie appeals to
# users who like dark, cerebral films" — purely from rating patterns.
# It never sees genre tags; it learns everything from user behavior.

print("\nTraining SVD model (this takes ~1–2 minutes)...")

reader = Reader(rating_scale=(0.5, 5.0))
data   = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

# Train on the FULL dataset (no train/test split here — we want the model
# to have seen as many movies as possible for recommendation quality)
trainset = data.build_full_trainset()

model = SVD(
    n_factors=100,    # Number of latent dimensions to learn
    n_epochs=20,      # Training iterations
    lr_all=0.005,     # Learning rate
    reg_all=0.02,     # Regularization (prevents overfitting)
    random_state=42
)
model.fit(trainset)

with open(SRC / "svd_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("✅ SVD model saved")


Movies  : 9,742
Ratings : 100,836
Users   : 610

✅ TF-IDF artifacts saved

Training SVD model (this takes ~1–2 minutes)...
✅ SVD model saved
